# Fine dropout in model 5

* How to identify common and uncommon tokens in a text dataset
* How LLMs adjust processing of tokens, even when they-re not currently being processed
* That softmax-based loss functions train the model on all tokens, not only the currently selected ones

# Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import requests

import torch
import torch.nn as nn
import torch.nn.functional as F

# vector plots
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

In [ ]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
# hyperparams for GPT2-124M
n_vocab = 50257
embed_dim = 768
seq_len = 256
n_heads = 12
n_blocks = 12
batch_size = 16

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Tokens Count in Corpus Text

In [ ]:
text = requests.get('http://www.gutenberg.org/cache/epub/829/pg829.txt').text
gt_tokens = torch.tensor(tokenizer.encode(text),dtype=torch.long)
len(gt_tokens)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (158340 > 1024). Running this sequence through the model will result in indexing errors


158340

In [ ]:
uniq, counts = np.unique(gt_tokens, return_counts = True)

print(f'{len(gt_tokens):,} total tokens, and {len(uniq):,} unique tokens.')
print(f"That's {100*len(uniq)/tokenizer.vocab_size:.1f}% of the tokenizer vocabulary")

158,340 total tokens, and 10,274 unique tokens.
That's 20.4% of the tokenizer vocabulary


# Frequencies of tokens

In [ ]:
freq_idx = np.argsort(counts)[::-1]

# most and least common
most_common = freq_idx[:100]
least_common = freq_idx[-100:]

# tokens in the dictionary that don't appear in the text
never_used = []
i = 0
while len(never_used)<100:
  if i not in gt_tokens:
    never_used.append(i)
  i+=1

In [ ]:
# print out some examples
print('20 MOST common tokens:')
for i in most_common[:20]:
    print(f' {counts[i]:4} occurances of "{tokenizer.decode(uniq[i])}"')

print('\n\n20 LEAST common tokens:')
for i in least_common[:20]:
    print(f' {counts[i]:4} occurance(s) of "{tokenizer.decode(uniq[i])}"')

print('\n\n20 NEVER used tokens:')
for i in never_used[:20]:
    print(f' {len(np.where(gt_tokens==i)[0])} occurances of "{tokenizer.decode(i)}"')

20 MOST common tokens:
 10060 occurances of ","
"
 9923 occurances of "
"
 5393 occurances of " the"
 3779 occurances of " of"
 3468 occurances of " and"
 3356 occurances of " to"
 3006 occurances of "."
 2657 occurances of " I"
 2431 occurances of " a"
 1900 occurances of " in"
 1562 occurances of ";"
 1474 occurances of " my"
 1199 occurances of " was"
 1061 occurances of " that"
  964 occurances of " me"
  941 occurances of " with"
  920 occurances of " as"
  823 occurances of " his"
  794 occurances of " by"


20 LEAST common tokens:
    1 occurance(s) of "esides"
    1 occurance(s) of " identification"
    1 occurance(s) of " Holl"
    1 occurance(s) of " dealt"
    1 occurance(s) of " diam"
    1 occurance(s) of "Whereas"
    1 occurance(s) of " whispering"
    1 occurance(s) of "wark"
    1 occurance(s) of "gain"
    1 occurance(s) of "absolute"
    1 occurance(s) of "sharp"
    1 occurance(s) of "uper"
    1 occurance(s) of "bring"
    1 occurance(s) of " redress"
    1 occuran

# Average Category Logsoftmax in a Random Model

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self):
    super().__init__()

    # number of attention heads
    self.num_heads = n_heads
    self.head_dim = embed_dim // n_heads

    # the three QKV weights matrices are init as ont and are split inside
    self.QKV = nn.Linear(embed_dim, 3*embed_dim, bias = True)

    # linear mixing after attention
    self.W0 = nn.Linear(embed_dim, embed_dim, bias = True)

  def forward(self, x):

    # sizes for later use
    B, T, E = x.shape # [batch, seq_len, embed_dim]

    # push data through QV in one concatenated matrix
    qkv = self.QKV(x) # [batch, sequence, 3*embed]
    q,k,v = torch.split(qkv, E, dim = 2) # each matrix is [B,T,E]

    # reshape to [B,T,n_heads, head_dim]
    # and then transpose to [B, n_heads, T, head_dim]
    q = q.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
    k = k.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
    v = v.view(B, T, self.num_heads, self.head_dim).transpose(1,2)

    # PyTorchs dot-product attention function handles multi-head shapes
    out = F.scaled_dot_product_attention(q,k,v,is_causal=True)

    # recombine heads: (B, n_heads, T, head_dim) -> [B,T,E]
    out = out.transpose(1,2).view(B,T,E)

    # finally, linearly mix the attention heads
    out = self.W0(out)

    return out


class TransformerBlock(nn.Module):
  def __init__(self):
    super().__init__()

    # attention subblock
    self.layer_norm_1 = nn.LayerNorm(embed_dim, eps = 1e-5)
    self.attn = MultiHeadAttention()

    # linear feedforward (MLP) subblock
    self.layer_norm_2 = nn.LayerNorm(embed_dim, eps = 1e-5)
    # 4x expansion, then back to embedding size
    self.mlp_1 = nn.Linear(embed_dim, 4*embed_dim, bias = True)
    self.gelu  = nn.GELU()
    self.mlp_2 = nn.Linear(4*embed_dim, embed_dim, bias = True)

  def forward(self, x):

    # attention
    x_att = self.layer_norm_1(x) # pre-attention normalization
    x_att = x + self.attn(x_att) # run through attention, then add pre-attention

    # MLP
    x_ff = self.layer_norm_2(x_att) # pre-MLP normalization
    x_ff = x_att + self.mlp_2(self.gelu(self.mlp_1(x_ff))) # adjustment

    return x_ff

In [ ]:
class LanguageModel(nn.Module):
  def __init__(self):
    super().__init__()

    # token + position embeddings
    self.wte = nn.Embedding(n_vocab, embed_dim) # token embedding
    self.wpe = nn.Embedding(seq_len, embed_dim) # position embedding

    # transformer blocks
    self.transformer_blocks = nn.Sequential(*[TransformerBlock() for _ in range(n_blocks)])

    # final layer_norm
    self.layer_norm_final = nn.LayerNorm(embed_dim, eps = 1e-5)

    # lm head, with weights tied to token embeddings
    self.final_head = nn.Linear(embed_dim, n_vocab, bias = False)
    self.final_head.weight = nn.Parameter(self.wte.weight)

    # weight inits
    self.apply(self.weight_inits)

  def weight_inits(self, module):

    # initialize nn.linear to normal with std=.02
    if isinstance(module, nn.Linear):
      nn.init.normal_(module.weight, mean = 0, std = .02)

      # initialize bias terms to zero
      if module.bias is not None:
        nn.init.zeros_(module.bias)

    # nn.Embeddings to Xavier
    if isinstance(module, nn.Embedding):
      nn.init.xavier_normal_(module.weight)

  def forward(self, idx):

    # token + position embeddings (note the device)
    token_emb = self.wte(idx) # [B,T,E]
    posit_emb = self.wpe(torch.arange(idx.shape[-1],device = device)) # [T,E]
    x = token_emb + posit_emb # [B,T,E]

    # pass through each transformer block
    x = self.transformer_blocks(x)

    # final layer_norm and unembeddings
    x = self.layer_norm_final(x)
    logits = self.final_head(x) # [B,T,n_vocab]

    # scale and logsoftmax
    outputs = F.log_softmax(logits/np.sqrt(embed_dim), dim = -1)

    return outputs

In [ ]:
model = LanguageModel().to(device)

In [ ]:
# generate a batch of data
ix = torch.randint(len(gt_tokens) - seq_len, size=(batch_size,))
x  = gt_tokens[ix[:, None] + torch.arange(seq_len)].to(device)

# forward pass
out = model(x).detach().cpu().numpy()

# create a submatrix of all outputs for this set of words
sub_matrix = out[:, :, most_common]

print(f'Size of input:     {x.shape}')
print(f'Size of output:    {out.shape}')
print(f'Size of submatrix: {sub_matrix.shape}')

Size of input:     torch.Size([16, 256])
Size of output:    (16, 256, 50257)
Size of submatrix: (16, 256, 100)


In [ ]:
# average all output values
most_ave  = out[:, :, most_common].mean()
least_ave = out[:, :, least_common].mean()
never_ave = out[:, :, never_used].mean()

print(f'{most_ave:.2f} for most common')
print(f'{least_ave:.2f} for least common')
print(f'{never_ave:.2f} for unused')

-10.82 for most common
-10.82 for least common
-10.82 for unused


In [ ]:
# reshape
x_flat = x.view(-1).cpu().numpy()

# extract a submatrix and flatten
sub_matrix = out[:, :, most_common]
sub_flat = sub_matrix.reshape(batch_size * seq_len, -1)

# filter using boolean indexing
mask = ~np.isin(x_flat, most_common)  # boolean mask for filtering
data2_ave = sub_flat[mask, :]         # select only non-common token outputs

print(f'Size of input:               {x_flat.shape}')
print(f'Size of output:              {out.shape}')
print(f'Size of submatrix:           {sub_matrix.shape}')
print(f'Size of flattened submatrix: {sub_flat.shape}')

Size of input:               (4096,)
Size of output:              (16, 256, 50257)
Size of submatrix:           (16, 256, 100)
Size of flattened submatrix: (4096, 100)


In [ ]:
# sanity-check that the mask has no high-frequency tokens
tokenizer.decode(x_flat[mask])

for m in most_common:
    print(m) if m in x_flat[mask] else None

# Track Category Log-Softmax During Training

In [ ]:
loss_function = nn.NLLLoss(reduction='mean').to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr = .001, weight_decay = True)

In [ ]:
num_samples = 501
token_probs = np.zeros((num_samples, 3))

# init losses
train_loss = []

for sampli in range(num_samples):

  # get a batch of data
  ix = torch.randint(len(gt_tokens)-seq_len, size = (batch_size,)) # we take random tokens from whole corpus of size batch_size
  X  = gt_tokens[ix[:,None] + torch.arange(seq_len)]    # we add empty dim by None, and to each selected token we add arange of seq_len to get following seq_len tokens for each selected token, so if randomly selected token is "I", we want following seq_len "I like to eat potatoes on a warm sand"
  y  = gt_tokens[ix[:,None] + torch.arange(1,seq_len+1)] # same for output we just move tokens 1 step right

  # move data to GPU
  X, y = X.to(device), y.to(device)

  # # clear previous gradients
  # model.zero_grad(set_to_none=True)

  # forward pass
  log_probs = model(X)

  # calculate the losses on the (reshaped) tokens
  loss = loss_function(log_probs.view(-1, log_probs.shape[-1]), y.view(-1))

  # backprop
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  # store the per-sample loss
  train_loss.append(loss.item())

  ## calculate outputs for token frequency category
  # flatten the model inputs and outputs
  X_flat = X.view(-1).cpu().numpy()
  log_probs = log_probs.detach().cpu().numpy()

  # Most common tokens: extract submatrix, and average tokens
  log_probs_filtered = log_probs[:,:,uniq[most_common]].reshape(batch_size*seq_len,-1) # [batch_num, seq_len, n_vocab] -> filter -> [batch_num, seq_len, most_common] -> [batch_num x seq_len, most_common]
  mask = ~np.isin(X_flat, uniq[most_common]) # [True, False, ...] list (mask) of tokens that are NOT in most_common
  token_probs[sampli, 0] = log_probs_filtered[:,:].mean()

  # LEAST COMMON tokens: probability model assigns to least_common tokens, when input is NOT least_common
  log_probs_filtered = log_probs[:, :, uniq[least_common]].reshape(batch_size * seq_len, -1)
  mask = ~np.isin(X_flat, uniq[least_common])
  token_probs[sampli, 1] = log_probs_filtered[:, :].mean()

  # UNUSED tokens: probability model assigns to never_used tokens, when input is NOT never_used
  log_probs_filtered = log_probs[:, :, uniq[never_used]].reshape(batch_size * seq_len, -1)
  mask = ~np.isin(X_flat, uniq[never_used])
  token_probs[sampli, 2] = log_probs_filtered[:, :].mean()

  # update progress display
  if sampli % 100 == 0:
    print(f'Sample {sampli:4}, train loss: {train_loss[-1]:.4f}')

Sample    0, train loss: 10.8236


In [ ]:
# plot the losses
plt.figure(figsize=(8, 4))
plt.plot(train_loss, 'k', markersize=8)
plt.gca().set(xlabel='Data batch', ylabel='Loss')
plt.show()

In [ ]:
# show the averages
plt.figure(figsize=(12, 4))

plt.plot(token_probs, linewidth=2)
plt.legend(['Most common', 'Least common', 'Never used'])
plt.gca().set(
    xlabel='Training sample',
    ylabel='Log softmax output',
    xlim=[0, num_samples]
)

plt.show()

# Breakdown

In [ ]:
# get a batch of data
ix = torch.randint(len(gt_tokens)-seq_len, size = (batch_size,)) # we take random tokens from whole corpus of size batch_size
X  = gt_tokens[ix[:,None] + torch.arange(seq_len)]    # we add empty dim by None, and to each selected token we add arange of seq_len to get following seq_len tokens for each selected token, so if randomly selected token is "I", we want following seq_len "I like to eat potatoes on a warm sand"
y  = gt_tokens[ix[:,None] + torch.arange(1,seq_len+1)] # same for output we just move tokens 1 step right

random tokens from corpus of length `batch_size`

In [ ]:
ix

Create matrix of n_tokens with arange seq_len of following tokens

In [ ]:
X  = gt_tokens[ix[:,None] + torch.arange(seq_len)]
y  = gt_tokens[ix[:,None] + torch.arange(1,seq_len+1)]

X.shape, y.shape

One row has `seq_len` following tokens where 1st token was selected randomly

In [ ]:
tokenizer.decode(X[0,:]), tokenizer.decode(y[0,:])

We move to device and return log_probs which are of shape [batch_size, seq_len, n_vocab], i.e. for each batch we have sequence and for each token in sequence we have log-probability distribution assembled out of 50257 potential tokens.

In [ ]:
# move data to GPU
X, y = X.to(device), y.to(device)

# forward pass
log_probs = model(X)
log_probs.shape

We calculate loss

In [ ]:
log_probs.shape[-1]

In [ ]:
log_probs.view(-1, log_probs.shape[-1]).shape, y.view(-1).shape

Above, for each [batch_size x seq_len] we have distribution of 50257 log-probabilities of next token, then we compare it with flatten y tokens which also have [batch_size x seq_len] length.

In [ ]:
# calculate the losses on the (reshaped) tokens
loss = loss_function(log_probs.view(-1, log_probs.shape[-1]), y.view(-1))
loss

We do backprop and append the losses

In [ ]:
# backprop
optimizer.zero_grad()
loss.backward()
optimizer.step()

# store the per-sample loss
train_loss.append(loss.item())

In [ ]:
## calculate outputs for token frequency category
# flatten the model inputs and outputs
X_flat = X.view(-1).cpu().numpy()
log_probs = log_probs.detach().cpu().numpy()

In [ ]:
log_probs.shape

We flatten out X to X_flat and push log_probs from device to cpu and call it outs.

In [ ]:
X_flat.shape, outs.shape

We filter the full n_vocab log-probability distribution, keeping only the 100 most_common tokens, meaning columns are assembled from log-probabilities of 100 most_common tokens

In [ ]:
log_probs[:,:,most_common].reshape(batch_size * seq_len, -1).shape

In [ ]:
# Most common tokens: extract submatrix, and average tokens
log_probs_filtered = log_probs[:,:,most_common].reshape(batch_size*seq_len,-1) # [batch_num, seq_len, n_vocab] -> filter -> [batch_num, seq_len, most_common] -> [batch_num x seq_len, most_common]
log_probs_filtered.shape

We create a mask for each row that tells us TRUE where token from randomly selected X isn't in most_common

In [ ]:
mask = ~np.isin(X_flat, most_common) # [True, False, ...] list (mask) of tokens that are NOT in most_common
mask, mask.shape

Then for all tokens that are in X_flat are are NOT in most_common we calculate mean of log_progbs to see "how likely are they can be selected during training."

In [ ]:
sub_matrix[mask,:].shape

In [ ]:
token_probs[sampli, 0] = sub_matrix[mask,:].mean()

In [ ]:
sub_matrix[mask,:].shape

In [ ]:
sub_matrix[mask,:].mean()

We do same thing for least_common tokens and never_used tokens

In [ ]:
# LEAST COMMON tokens: extract submatrix, and average tokens not present
sub_matrix = outs[:, :, least_common].reshape(batch_size * seq_len, -1)
mask = ~np.isin(X_flat, least_common)
token_probs[sampli, 1] = sub_matrix[mask, :].mean()

# UNUSED tokens: extract submatrix, and average tokens not present
sub_matrix = outs[:, :, never_used].reshape(batch_size * seq_len, -1)
mask = ~np.isin(X_flat, never_used)
token_probs[sampli, 2] = sub_matrix[mask, :].mean()

# Conclusion

* Softmax affects all token logits, not just the ones currently being processed, all tokens embeddings are trained inside the model, not just the ones that appear in the current token sequence
* This has implications for model output coherence and accuracy for tokens that are less commonly used in training datasets (e.g., obscure topics, new coding lnaguages)